In [0]:
import pyspark.sql.functions as F

# Define parameters so a job can call the simulator
dbutils.widgets.text("catalog", "meridian_tNN")
dbutils.widgets.text("batch_number", "1")
dbutils.widgets.text("rows_per_batch", "2000")

catalog = dbutils.widgets.get("catalog")
batch = int(dbutils.widgets.get("batch_number"))
rows = int(dbutils.widgets.get("rows_per_batch"))

volume_path = f"/Volumes/{catalog}/landing/raw_files"
dimensions = ["reason", "store", "item", "customer"]

# Land dimensions and lookup table once during batch 1
if batch == 1:
    for dim in dimensions:
        df_dim = spark.table(f"samples.tpcds_sf1.{dim}")
        df_dim.write.mode("overwrite").parquet(f"{volume_path}/{dim}/batch_1")
    
    # Store sales is a read-only lookup
    df_sales = spark.sql("SELECT * FROM samples.tpcds_sf1.store_sales LIMIT 10000")
    df_sales.write.mode("overwrite").parquet(f"{volume_path}/store_sales/batch_1")

# Slice fact data (store_returns) into disjoint batches
offset = (batch - 1) * rows
df_fact = spark.sql(f"""
    SELECT * FROM samples.tpcds_sf1.store_returns 
    ORDER BY sr_returned_date_sk 
    LIMIT {rows} OFFSET {offset}
""")

# Inject bad data (null keys, duplicates, invalid values) for Silver to catch later
df_bad_data = df_fact.limit(3) \
    .withColumn("sr_item_sk", F.lit(None).cast("int")) \
    .withColumn("sr_return_quantity", F.lit(-5))

df_final_batch = df_fact.union(df_bad_data)

# Write the fact batch
df_final_batch.write.mode("overwrite").parquet(f"{volume_path}/store_returns/batch_{batch}")